# SpatialScan — real pipeline on Colab's free T4 GPU

Turns a short video you filmed into a walkable 3D `.splat` scene using the **real** pipeline:
ffmpeg → COLMAP → 3D Gaussian Splatting (nerfstudio `splatfacto`) → `.splat` export.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Filming tips:** ≤45 seconds, move slowly sideways (not just rotating in place), overlap what you film, good lighting, avoid blank textureless walls.

In [ ]:
# 1. Verify the GPU
!nvidia-smi

In [ ]:
# 2. System dependencies: COLMAP + ffmpeg (~2 min)
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!colmap -h | head -3 && ffmpeg -version | head -1

In [ ]:
# 3. Python dependencies: nerfstudio brings torch-compatible gsplat (~5-8 min)
%pip -q install nerfstudio
!ns-train --help | head -3

In [ ]:
# 4. Get the SpatialScan worker
!git clone https://github.com/Gilhzn/Parallax-AI.git
%pip -q install -e Parallax-AI/services/worker

In [ ]:
# 5. Upload your video (mp4/mov, up to ~45s)
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))
print('will process:', video_path)

In [ ]:
# 6. Run the real pipeline
#    QUALITY presets:
#      'fast'     ~3-5 min on T4  - quick preview
#      'balanced' ~10-20 min on T4 - great quality (recommended on free T4)
#      'high'     ~45-90 min on T4 - FULL resolution, sharp-frame selection,
#                 splatfacto-big @ 30k iterations, keeps lossless PLY.
#                 (On an A100/4090 'high' takes ~10-20 min.)
QUALITY = 'high'  # <- change to 'balanced' if the free T4 disconnects

!cd Parallax-AI/services/worker && python -m spatialscan_worker.cli \
    --video "/content/{video_path}" \
    --out /content/out \
    --mode real \
    --quality {QUALITY}


In [ ]:
# 7. Download the results
import os
from google.colab import files
files.download('/content/out/scene.splat')
files.download('/content/out/manifest.json')
if os.path.exists('/content/out/scene.ply'):
    files.download('/content/out/scene.ply')  # lossless archive (quality=high)


## Viewing the result

- In the SpatialScan web app: replace `apps/web/public/sample.splat` with your `scene.splat`, run `npm run dev`, open `http://localhost:5173/tour/sample`.
- Or any online antimatter15-format `.splat` viewer.

If COLMAP registers too few frames (error in the poses stage): film again with slower, wider motion and more texture in view; or raise `--fps` to 4-5.